In [2]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv


In [23]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

train_df.head()

,fen,best
0,6qn/p5kp/PpQ2rp1/2n1NbP1/1PP2P1P/B1P1P3/8/7K w...,c6f6
1,rnb1r3/qp3k2/3bp2p/p1pP1p2/1PP2P2/P2B1nQ1/3P1N...,g3g7
2,4k3/R7/4K3/2BP4/6P1/7p/7N/4R3 w - - 6 106,a7a8
3,4k3/3r3p/2ppB2P/bP3Qp1/5RP1/PP6/5B2/7K w - - 3 59,f5f8
4,3k4/5P2/1p2P1p1/r3p2n/P3Pbq1/3b4/8/2R4K w - - ...,f7f8q



Valores relativos de las piezas de ajedrez

| Pieza   | Blancas | Negras |
| ------- | ------- | ------ |
| Peon    | 1       | -1     |
| Caballo | 3       | -3     |
| Alfil   | 4       | -4     |
| Torre   | 5       | -5     |
| Reina   | 9       | -9     |
| Rey     | 10      | -10    |


In [43]:
PIECE_MAP = {
    'P':  1, 'N':  3, 'B':  4, 'R':  5, 'Q':  9, 'K':  10,
    'p': -1, 'n': -3, 'b': -4, 'r': -5, 'q': -9, 'k': -10
}

def fen_to_board(fen: str) -> np.ndarray:
    board_part = fen.split()[0]  # Parte del tablero
    rows = board_part.split('/') # 8 filas

    # Crea el tablero con las piezas
    board = np.zeros((8, 8), dtype=np.float32)
    for i, row in enumerate(rows):
        col = 0
        for chess_piece in row:
            if chess_piece.isdigit():
                col += int(chess_piece) # casillas vacías
            else:
                board[i, col] = PIECE_MAP[chess_piece]
                col += 1
    return board

def fen_to_metadata(fen: str) -> np.ndarray:
    parts = fen.split()

    # Turno: blancas = 1, negras = -1
    turn = 1.0 if parts[1] == 'w' else -1.0

    # Enroque: variables binarias (K, Q, k, q)
    castling = parts[2] if len(parts) > 2 else '-'
    K = float('K' in castling)
    Q = float('Q' in castling)
    k = float('k' in castling)
    q = float('q' in castling)

    # En passant: 1 si hay casilla disponible, 0 si no
    en_passant = parts[3] if len(parts) > 3 else '-'
    ep = 0.0 if en_passant == '-' else 1.0

    return np.array([turn, K, Q, k, q, ep], dtype=np.float32)

# Prueba
sample_fen = train_df.iloc[50]['fen']
print(fen_to_board(sample_fen))
print(fen_to_metadata(sample_fen))

[[  0.  -5.  -4.  -9. -10.  -4.  -3.   0.]
 [  0.   0.  -1.  -1.  -1.   0.   1.  -5.]
 [ -1.   0.   0.   0.   0.   0.   0.  -1.]
 [  1.  -1.  -3.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   4.   3.]
 [  0.   0.   0.   0.   0.   1.   0.   0.]
 [  0.   1.   1.   1.   1.   0.   0.   1.]
 [  5.   3.   4.   9.  10.   0.   0.   5.]]
[1. 1. 1. 0. 0. 0.]


In [52]:
MAX_PIECE_VALUE = 10.0 # Valor del rey

# Convierte el tablero en un vector de 70 casillas
def fen_to_features(fen: str) -> np.ndarray:
    # Escala los vectores en [1, -1]
    board    = fen_to_board(fen).flatten() / MAX_PIECE_VALUE  # 64 casillas
    metadata = fen_to_metadata(fen)                           # 6 metadatas
    return np.concatenate([board, metadata])

# Lo aplica a todo el dataset
X_train = np.array([fen_to_features(fen) for fen in train_df['fen']])
X_test  = np.array([fen_to_features(fen) for fen in test_df['fen']])

sample_vec = fen_to_features(sample_fen)